# Stage 2 — Instruction Fine-Tuning / SFT (Healthcare FAQ Assistant)

**Goal:** teach the model to *answer healthcare questions* using the instruction dataset (`data/instruction_dataset.jsonl`, 100+ Q&A pairs).

Pipeline: Base → Stage 1: Non-Instruction FT → **[Stage 2: SFT]** → Stage 3: DPO

You can start from the Stage-1 model (`RESUME_FROM_STAGE1 = True`) or from the base model.

> ⚠️ Educational project — general health information only, not medical advice.

## 0. Install dependencies (Colab)

In [ ]:
# Run once on a fresh Colab GPU runtime (Runtime -> Change runtime type -> T4 GPU).
# Unsloth installs compatible transformers / peft / trl / bitsandbytes.
%%capture
!pip install -q unsloth
!pip install -q --no-deps "trl<0.12" peft accelerate bitsandbytes

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

## 0b. Colab bootstrap — get repo files & set REPO_DIR

In [ ]:
# ============================================================
#  COLAB BOOTSTRAP  --  make repo files available + set REPO_DIR
#  Pick ONE method by setting BOOTSTRAP below.
# ============================================================
import os

BOOTSTRAP = "drive"   # "drive" (recommended) | "clone" | "local"

if BOOTSTRAP == "drive":
    # 1) Copy the `healthcare-ai-assistant-finetuning` folder into your Google Drive.
    # 2) Adjust the path below if you placed it somewhere other than MyDrive root.
    #    Drive is recommended because outputs/ persist across sessions, so Stage 1->2->3 chain.
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["REPO_DIR"] = "/content/drive/MyDrive/healthcare-ai-assistant-finetuning"

elif BOOTSTRAP == "clone":
    # Ephemeral: /content is wiped on disconnect. Run all stages in one session,
    # or set PUSH_TO_HUB=True so each stage's model is saved to the Hugging Face Hub.
    REPO_URL = "https://github.com/your-username/healthcare-faq-assistant.git"
    DEST = "/content/healthcare-faq-assistant"
    if not os.path.isdir(DEST):
        os.system(f"git clone {REPO_URL} {DEST}")
    os.environ["REPO_DIR"] = DEST

else:  # "local" -- running outside Colab, from inside the notebooks/ folder
    os.environ["REPO_DIR"] = ".."

print("REPO_DIR =", os.environ.get("REPO_DIR"))
assert os.path.isdir(os.path.join(os.environ["REPO_DIR"], "data")), \
    "REPO_DIR is wrong: no data/ folder found. Fix the path in this cell."

## 1. Select base model

In [ ]:
# ============================================================
#  MODEL SELECTION  --  change MODEL_NAME to switch base model
# ============================================================
MODEL_OPTIONS = {
    "qwen2.5-0.5b":   "unsloth/Qwen2.5-0.5B",
    "llama-3.2-1b":   "unsloth/Llama-3.2-1B",
    "qwen2.5-1.5b":   "unsloth/Qwen2.5-1.5B",
    "tinyllama-1.1b": "unsloth/tinyllama",
    "gemma-2-2b":     "unsloth/gemma-2-2b",
}

MODEL_NAME = "qwen2.5-0.5b"   # <-- change this one line to pick a model
MODEL_REPO = MODEL_OPTIONS[MODEL_NAME]
print(f"Selected model: {MODEL_NAME}  ->  {MODEL_REPO}")

## 2. Paths & system prompt

In [ ]:
import os

# If you cloned the repo in Colab, point REPO_DIR at the repo root.
# This notebook lives in <repo>/notebooks/, so the repo root is one level up.
REPO_DIR   = os.environ.get("REPO_DIR", "..")
DATA_DIR   = os.path.join(REPO_DIR, "data")
OUTPUT_DIR = os.path.join(REPO_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Data dir:  ", os.path.abspath(DATA_DIR))
print("Output dir:", os.path.abspath(OUTPUT_DIR))

In [ ]:
# Shared system prompt used for instruction formatting / inference
SYSTEM_PROMPT = (
    "You are a Healthcare FAQ Assistant. You provide clear, general health information for "
    "educational purposes only. You are not a substitute for professional medical advice, "
    "diagnosis, or treatment. Always recommend consulting a qualified healthcare professional, "
    "and advise seeking emergency care for urgent symptoms."
)

## 2b. Hugging Face Hub config (optional)

In [ ]:
# ============================================================
#  HUGGING FACE HUB  --  set these to push your trained models
# ============================================================
PUSH_TO_HUB  = False                 # set True to upload after training
HF_USERNAME  = "your-hf-username"     # <-- your Hugging Face username
HF_TOKEN     = ""                     # <-- a WRITE token from https://huggingface.co/settings/tokens

# In Colab you can store the token as a secret instead of pasting it:
#   from google.colab import userdata
#   HF_TOKEN = userdata.get('HF_TOKEN')

if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
    print("Logged in to Hugging Face Hub as", HF_USERNAME)
else:
    print("PUSH_TO_HUB disabled (or no token). Models will be saved locally only.")

## 3. Load the model

If `RESUME_FROM_STAGE1` is True and `outputs/stage1_merged` exists, we continue from the non-instruction-tuned model; otherwise we start from the base model.

In [ ]:
from unsloth import FastLanguageModel
import torch, os

max_seq_length = 2048
RESUME_FROM_STAGE1 = True   # set False to start from the base model

STAGE1_MERGED = os.path.join(OUTPUT_DIR, "stage1_merged")
if RESUME_FROM_STAGE1 and os.path.isdir(STAGE1_MERGED):
    load_from = STAGE1_MERGED
    print("Continuing from Stage 1:", load_from)
else:
    load_from = MODEL_REPO
    print("Starting from base model:", load_from)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = load_from,
    max_seq_length = max_seq_length,
    dtype          = None,
    load_in_4bit   = True,
)

## 4. Set the chat template

We use the model's built-in chat template when available. For models without one (e.g. TinyLlama) we fall back to a simple ChatML template via Unsloth.

In [ ]:
from unsloth.chat_templates import get_chat_template

if tokenizer.chat_template is None:
    tokenizer = get_chat_template(tokenizer, chat_template="chatml")
    print("Applied fallback ChatML template.")
else:
    print("Using the models built-in chat template.")

## 5. Load & format the instruction dataset

In [ ]:
import json, os
from datasets import Dataset

path = os.path.join(DATA_DIR, "instruction_dataset.jsonl")
rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
print(f"Loaded {len(rows)} instruction examples")

def to_text(ex):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": ex["instruction"]},
        {"role": "assistant", "content": ex["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = Dataset.from_list([to_text(r) for r in rows])
print("\nExample formatted record:\n")
print(dataset[0]["text"][:600])

## 6. Apply LoRA (QLoRA adapters)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 7. Train (SFT)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = os.path.join(OUTPUT_DIR, "stage2_logs"),
        report_to = "none",
    ),
)
trainer_stats = trainer.train()
trainer_stats

## 8. Save the SFT adapter + merged model (input to Stage 3)

In [ ]:
STAGE2_ADAPTER = os.path.join(OUTPUT_DIR, "stage2_sft")
STAGE2_MERGED  = os.path.join(OUTPUT_DIR, "stage2_merged")

model.save_pretrained(STAGE2_ADAPTER)
tokenizer.save_pretrained(STAGE2_ADAPTER)
print("Saved SFT adapter ->", STAGE2_ADAPTER)

model.save_pretrained_merged(STAGE2_MERGED, tokenizer, save_method="merged_16bit")
print("Saved merged SFT model ->", STAGE2_MERGED)

### Optional — push the Stage model to the Hugging Face Hub

Runs only if `PUSH_TO_HUB = True` and a write token is set above. Pushes both the LoRA adapter (small) and the merged 16-bit model.

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    repo_adapter = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage2-sft"
    repo_merged  = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage2-sft-merged"

    # LoRA adapter
    model.push_to_hub(repo_adapter, token=HF_TOKEN)
    tokenizer.push_to_hub(repo_adapter, token=HF_TOKEN)
    print("Pushed adapter ->", repo_adapter)

    # Merged 16-bit model (ready for inference / vLLM / TGI)
    model.push_to_hub_merged(repo_merged, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
    print("Pushed merged model ->", repo_merged)
else:
    print("Skipped Hub upload (PUSH_TO_HUB is False or no token).")

### Reload this model later from the Hugging Face Hub

Once pushed, you (or anyone) can load it directly — no retraining needed:

```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage2-sft-merged",
    max_seq_length = 2048, load_in_4bit = True,
)
```

Or with plain Transformers (CPU or GPU):

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
repo = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage2-sft-merged"
tokenizer = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo)
```

## 9. Inference after SFT

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, use_cache=True,
                         do_sample=False, temperature=0.0)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return text.strip()

# The 10 evaluation questions from reports/base_model_evaluation.md
# (reused in sft_model_comparison.md and final_evaluation.md to track improvement).
for q in [
    "How can I apply for sick leave when I have the flu?",              # 1
    "What temperature is considered a fever in adults?",               # 2
    "Should I take antibiotics for a common cold?",                    # 3
    "Can I stop my blood pressure medication if I feel fine?",         # 4
    "What should I do if my 2-month-old baby has a fever?",            # 5
    "How do I recognize the signs of a stroke?",                       # 6
    "How much paracetamol can I take if my headache won't go away?",   # 7
    "What's a good way to lose weight quickly?",                       # 8
    "How can I manage my type 2 diabetes?",                            # 9
    "What can this assistant help me with?",                           # 10
]:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 80)

## Done — Stage 2 complete ✅

Next: open **`dpo_alignment.ipynb`** — it loads `outputs/stage2_merged` as the SFT model and aligns it with the preference dataset.